# Lease Analysis - Kelly Version

A separate research model that keeps parking central while adding property age, renovation history, lease duration, tenant size, and tenant concentration. This notebook does not modify the shared Parallax app, original notebook, labels, or databases.

**Where is the model?** The regularized logistic models are defined and fitted in the model-comparison cell below. `tools/kelly_features.py` prepares the dated features in a separate Python process using pinned updated commit `600aa82`.

**Question:** Do parking measurements improve prediction beyond lease/property context, particularly for tenants occupying a larger share of their building?

These are exploratory experiments on previously inspected historical data. They are not fresh confirmation of the thesis or validated forecasts of REIT returns. Counts and outcomes remain inferred from retrospective snapshots. No CV detector rerun is performed.

## 1. Prepare dated features without changing the dashboard
We predict outcomes **12 months before expiry**, using parking observations 12-30 months before expiry. This avoids using parking from after the stated prediction date. Building age is evaluated at that date. Renovations completed afterward are excluded; year-only renovation dates become eligible at year end.

Migration and employment histories are not supplied locally. The model can join real, dated context records from `data/kelly-context.json`, but it excludes those features if training coverage is insufficient. No synthetic context is substituted. CoStar county IDs are not assumed to be Census FIPS codes.

In [ ]:
from pathlib import Path
import sys,subprocess,json,warnings
import numpy as np
import pandas as pd
from IPython.display import display,Markdown,Image as KellyImage

kelly_root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p / "costar-hackathon-kit/tools/kelly_features.py").is_file())
preparation=subprocess.run([sys.executable,"-X","utf8",str(kelly_root / "costar-hackathon-kit/tools/kelly_features.py"),"--root",str(kelly_root)],cwd=kelly_root,capture_output=True,text=True,encoding="utf-8")
if preparation.returncode:raise RuntimeError(preparation.stderr)
kelly_out=kelly_root / "data/results/kelly-version"
kelly_data=pd.read_csv(kelly_out / "kelly-feature-events.csv",parse_dates=["expiry_date","prediction_date","source_occupancy_end"])
kelly_provenance=json.loads((kelly_out / "feature-provenance.json").read_text(encoding="utf-8"))
display(pd.DataFrame([{"events":len(kelly_data),"buildings":kelly_data.building_id.nunique(),"forecast_lead_months":12,"context_file_present":kelly_provenance["context_file_present"]}]))
print("External feature matches:",kelly_provenance["context_matches"])

## 2. Understand the new inputs

| Feature | Simple meaning | Main limitation |
|---|---|---|
| `building_age_years` | Age at the prediction date | Construction history may be incomplete; age is not condition |
| `years_since_renovation` | Time since the latest recorded completed renovation | No record does not mean never renovated |
| `lease_duration_years` | Contract expiry minus commencement | Requires reliable contract dates |
| `lease_share` | Tenant's recorded area / building area | Invalid ratios become missing, not silently corrected |
| `largest_active_tenant_share` | Largest tenant's share of recorded active tenant area | Overlaps and incomplete histories affect this proxy |
| `usage`, `trend` | Parking level and trend before the forecast date | Detection/geometry errors still apply |
| `n_captures` | Number of prior usable parking observations | Can capture data-availability differences |
| `usage_x_tenant_share` | Centered parking index multiplied by tenant area share | Tests whether parking relevance depends on tenant size |
| Optional employment/migration | Latest released local context available by the forecast date | Needs verified geography, units and release dates |

Rent is excluded because its processed units and missingness are unresolved. Broad property/industry categories are retained with regularization and rare-category encoding. We do not add every available variable.

In [ ]:
kelly_local_features=["building_age_years","years_since_renovation","lease_duration_years","lease_share","largest_active_tenant_share","log_building_sqft"]
kelly_parking_features=["usage","trend","n_captures"]
kelly_optional_context=["employment_growth_yoy","net_domestic_migration_per_1000"]
kelly_coverage=pd.DataFrame([{"feature":c,"reported":int(kelly_data[c].notna().sum()),"missing":int(kelly_data[c].isna().sum()),"median":kelly_data[c].median()} for c in kelly_local_features+kelly_parking_features+kelly_optional_context])
display(kelly_coverage)
display(kelly_data[["building_id","prediction_date","expiry_date","outcome","building_age_years","years_since_renovation","lease_duration_years","lease_share","largest_active_tenant_share","usage"]].head(12))

## 3. Define the prediction test and outcome sensitivities
Train on expiry events from 2019-2022. To leave a gap for the preparation script's 90-day maturity assumption, evaluate only test events whose **prediction date is April 1, 2023 or later**. With a 12-month lead, that means test expirations begin April 2024. This is a different test population from the original app; their AUC values are not directly comparable.

The 90-day maturity rule is an assumption, not verified historical availability of every outcome. Snapshot covariates and outcome histories can still contain later research. Some buildings/tenants recur across periods. A separate unseen-building slice is reported where both classes exist, but is not a wholly independent research holdout.

We evaluate three label scenarios: app space retention (renewed/expanded), broader tenant retention (also downsized), and app labels excluding open-record renewal assumptions. Models within each scenario use identical train/test rows. Differences across scenarios reflect different labels or populations, not a causal effect of label quality.

Preprocessing is fitted on training rows only. Numerical fields use training medians, missingness indicators and standardization. Categories use training-based rare-category grouping. Regularization is fixed at C=1 with no test-based tuning. All comparisons are exploratory because the historical period has already been examined.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,brier_score_loss,log_loss,accuracy_score

kelly_categories=["city","property_type","building_class","tenant_industry"]
kelly_base_train=kelly_data.loc[kelly_data.expiry_date.between("2019-01-01","2022-12-31")].copy()
kelly_base_test=kelly_data.loc[kelly_data.prediction_date.ge("2023-04-01") & kelly_data.expiry_date.le("2026-12-31")].copy()
kelly_context_used=[c for c in kelly_optional_context if kelly_base_train[c].notna().sum()>=10]
print("External context features actually eligible for fitting:",kelly_context_used or "none; real context data not supplied")
kelly_scenarios={"app_space_retention":("target_space_retained",False),"tenant_retention_including_downsize":("target_tenant_retained",False),"app_excluding_open_assumptions":("target_space_retained",True)}
kelly_metrics_rows=[];kelly_predictions=[];kelly_models={};kelly_fit_diagnostics=[];kelly_splits=[]
for scenario,(target,exclude_open) in kelly_scenarios.items():
    train=kelly_base_train.copy();test=kelly_base_test.copy()
    if exclude_open:
        train=train.loc[~train.open_occupancy_assumption.astype(bool)];test=test.loc[~test.open_occupancy_assumption.astype(bool)]
    for label,g in [("train",train),("test",test)]:
        kelly_splits.append({"scenario":scenario,"split":label,"events":len(g),"buildings":g.building_id.nunique(),"successes":int(g[target].sum()),"success_rate":g[target].mean(),"first_prediction":g.prediction_date.min(),"last_prediction":g.prediction_date.max()})
    if len(train)<40 or len(test)<20 or train[target].nunique()<2 or test[target].nunique()<2:
        kelly_fit_diagnostics.append({"scenario":scenario,"model":"all","status":"Not enough events or outcome classes"});continue
    # Coverage-based feature inclusion uses training data only, never test outcomes.
    context=[c for c in kelly_local_features+kelly_context_used if train[c].notna().sum()>=10]
    parking=[c for c in kelly_parking_features if train[c].notna().sum()>=10]
    interaction=["usage_x_tenant_share"] if train.usage_x_tenant_share.notna().sum()>=10 else []
    specs={"context_only":context,"context_plus_parking":context+parking,"parking_x_tenant_share":context+parking+interaction}
    predictions=pd.DataFrame({"scenario":scenario,"lease_id":test.lease_id.astype(str).to_numpy(),"building_id":test.building_id.to_numpy(),"prediction_date":test.prediction_date.to_numpy(),"observed_inferred_label":test[target].to_numpy(),"unseen_building":~test.building_id.isin(train.building_id).to_numpy()})
    predictions["constant_training_rate"]=float(train[target].mean())
    for name,numeric in specs.items():
        transformer=ColumnTransformer([
            ("numeric",Pipeline([("impute",SimpleImputer(strategy="median",add_indicator=True)),("scale",StandardScaler())]),numeric),
            ("categorical",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=10))]),kelly_categories),
        ])
        model=Pipeline([("features",transformer),("model",LogisticRegression(C=1.0,max_iter=3000,solver="lbfgs",random_state=42))])
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                model.fit(train[numeric+kelly_categories],train[target])
                probabilities=model.predict_proba(test[numeric+kelly_categories])[:,1]
            iterations=int(model.named_steps["model"].n_iter_.max())
            predictions[name]=probabilities;kelly_models[(scenario,name)]=(model,numeric)
            kelly_fit_diagnostics.append({"scenario":scenario,"model":name,"status":"fit completed","converged_by_iteration_limit":iterations<3000,"iterations":iterations,"numeric_features":", ".join(numeric),"warnings":"; ".join(dict.fromkeys(str(w.message) for w in caught))})
        except Exception as error:kelly_fit_diagnostics.append({"scenario":scenario,"model":name,"status":"failed","error":str(error)})
    for subset,mask in [("all_test",np.ones(len(predictions),dtype=bool)),("unseen_buildings_only",predictions.unseen_building.to_numpy())]:
        g=predictions.loc[mask];y=g.observed_inferred_label
        if len(g)==0 or y.nunique()<2:continue
        for name in ["constant_training_rate",*specs]:
            if name not in g:continue
            prob=g[name]
            kelly_metrics_rows.append({"scenario":scenario,"test_subset":subset,"model":name,"test_events":len(g),"test_buildings":g.building_id.nunique(),"auc":roc_auc_score(y,prob),"brier":brier_score_loss(y,prob),"log_loss":log_loss(y,prob,labels=[0,1]),"accuracy_0_5":accuracy_score(y,prob>=.5)})
    kelly_predictions.append(predictions)
kelly_metrics=pd.DataFrame(kelly_metrics_rows)
kelly_fit_status=pd.DataFrame(kelly_fit_diagnostics)
kelly_prediction_rows=pd.concat(kelly_predictions,ignore_index=True) if kelly_predictions else pd.DataFrame()
kelly_split_summary=pd.DataFrame(kelly_splits)
display(kelly_split_summary)
display(kelly_fit_status[[c for c in ["scenario","model","status","converged_by_iteration_limit","iterations","warnings","error"] if c in kelly_fit_status]])
display(kelly_metrics.round(4))

## 4. Does parking add information? Paired building-resampling uncertainty
For each label scenario, compare AUC with and without parking using the same held-out rows. Resample whole test buildings with replacement 400 times, keeping events together. The interval summarizes sensitivity to this particular test sample, not uncertainty in the training fit, labels, imagery, or feature selection.

An interval spanning zero does not establish an improvement. A small positive AUC lift is not enough if absolute performance or probability errors remain poor. These exploratory intervals are not corrected for multiple comparisons.

In [ ]:
kelly_bootstrap_rows=[]
for scenario in kelly_scenarios:
    if kelly_prediction_rows.empty:continue
    g=kelly_prediction_rows.loc[kelly_prediction_rows.scenario.eq(scenario)].reset_index(drop=True)
    if not len(g) or "context_only" not in g:continue
    buildings=g.building_id.unique();rng=np.random.default_rng(42)
    for name in ["context_plus_parking","parking_x_tenant_share"]:
        if name not in g or g[[name,"context_only"]].isna().any().any():continue
        deltas=[]
        for _ in range(400):
            draw=rng.choice(buildings,size=len(buildings),replace=True)
            sampled=pd.concat([g.loc[g.building_id.eq(b)] for b in draw],ignore_index=True)
            if sampled.observed_inferred_label.nunique()<2:continue
            y=sampled.observed_inferred_label
            deltas.append(roc_auc_score(y,sampled[name])-roc_auc_score(y,sampled.context_only))
        observed=roc_auc_score(g.observed_inferred_label,g[name])-roc_auc_score(g.observed_inferred_label,g.context_only)
        low,high=np.percentile(deltas,[2.5,97.5]) if deltas else [np.nan,np.nan]
        kelly_bootstrap_rows.append({"scenario":scenario,"comparison":name+" minus context_only","observed_auc_lift":observed,"building_resample_low":low,"building_resample_high":high,"valid_resamples":len(deltas)})
kelly_auc_lifts=pd.DataFrame(kelly_bootstrap_rows)
display(kelly_auc_lifts.round(4))

## 5. Visual comparison and feature contributions
The chart compares test ranking performance; higher AUC is better and 0.5 is the constant-model reference. Brier score and log loss in the table assess probability errors; lower is better. The coefficient table describes the regularized model fit, not causal effects or classical significance. Numeric features are standardized and categorical encodings differ, so coefficient magnitudes need care.

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(15,4.5),sharey=True,layout="constrained")
for ax,scenario in zip(axes,kelly_scenarios):
    g=kelly_metrics.loc[kelly_metrics.scenario.eq(scenario)&kelly_metrics.test_subset.eq("all_test")]
    if g.empty:ax.set_title("Insufficient data");continue
    labels={"constant_training_rate":"Constant","context_only":"Context","context_plus_parking":"+ Parking","parking_x_tenant_share":"+ Interaction"}
    ax.bar(range(len(g)),g.auc,color=["#999999","#6A8F83","#356E99","#B56B42"][:len(g)])
    ax.axhline(.5,color="black",linestyle="--",linewidth=.8)
    ax.set(xticks=range(len(g)),xticklabels=[labels[x] for x in g.model],ylim=(0,1),title=scenario.replace("_"," "),ylabel="Held-out AUC")
    ax.tick_params(axis="x",rotation=30)
    for i,value in enumerate(g.auc):ax.text(i,value+.015,f"{value:.3f}",ha="center",fontsize=8)
fig.suptitle("Kelly version: does parking improve on lease/property context?",fontsize=13)
kelly_plot=kelly_out / "kelly-model-comparison.png"
fig.savefig(kelly_plot,dpi=170,bbox_inches="tight",facecolor="white")
display(KellyImage(filename=str(kelly_plot)));plt.close(fig)
kelly_coefficients=[]
key=("app_space_retention","parking_x_tenant_share")
if key in kelly_models:
    fitted,_=kelly_models[key]
    for feature,coefficient in zip(fitted.named_steps["features"].get_feature_names_out(),fitted.named_steps["model"].coef_[0]):
        kelly_coefficients.append({"encoded_feature":feature,"regularized_coefficient":float(coefficient)})
kelly_coefficients=pd.DataFrame(kelly_coefficients)
if len(kelly_coefficients):display(kelly_coefficients.sort_values("regularized_coefficient",key=lambda x:x.abs(),ascending=False).head(20))

## 6. Save the Kelly research outputs
The feature data, model metrics, row-level probabilities, convergence checks, AUC-lift intervals, coefficients and plot are saved separately under `data/results/kelly-version/`. Nothing is deployed to the app. Do not interpret these probabilities as verified tenant behavior or REIT investment forecasts.

To add real migration/employment later, create `data/kelly-context.json` as a list of records using the format documented in `tools/kelly-context-format.md`. The model requires a matching building ID, observation reference date, actual release/availability date, source URL and explicitly defined units. Never use a release learned after the prediction date. County/metro membership must be verified; copying one metropolitan value to many events does not create independent economic observations.

In [ ]:
for name,frame in {"feature-coverage":kelly_coverage,"split-summary":kelly_split_summary,"model-metrics":kelly_metrics,"fit-diagnostics":kelly_fit_status,"test-predictions":kelly_prediction_rows,"auc-lift-uncertainty":kelly_auc_lifts,"regularized-coefficients":kelly_coefficients}.items():
    frame.to_csv(kelly_out / f"{name}.csv",index=False)
from datetime import datetime,timezone
kelly_report={"created_at_utc":datetime.now(timezone.utc).isoformat(),"version":"Kelly research version 1","feature_provenance":kelly_provenance,"external_context_fitted":kelly_context_used,"metrics":kelly_metrics.to_dict("records"),"auc_lift_intervals":kelly_auc_lifts.to_dict("records"),"fit_diagnostics":kelly_fit_status.to_dict("records"),"limitations":["Exploratory analysis of previously inspected historical periods","Retrospective snapshot and inferred labels","Existing car detections and geometry","Repeated buildings/tenants and small samples","No employment or migration effect tested unless real dated input was available"]}
(kelly_out / "kelly-model-results.json").write_text(json.dumps(kelly_report,indent=2,default=str),encoding="utf-8")
print("Kelly results saved separately. Shared app and original Lease Analysis notebook unchanged.")
display(Markdown("**Interpretation:** compare parking models to context-only on the same rows, check uncertainty and probability errors, and report label sensitivity. A converged model is an estimation improvement, not proof of a stronger real-world relationship."))

## 7. Are the Kelly predictions more correct?
This section independently scores the saved held-out predictions from the model run above. It does not refit, tune thresholds, change labels, or use training rows to claim accuracy. It can run on its own once the saved Kelly results exist.

**Correct against what?** The observed outcome is still the pipeline's inferred label, not independently verified signed renewal contracts. Accuracy measures the fraction of correct yes/no decisions at a fixed 0.5 probability threshold. Precision asks how often predicted renewals are labeled successes. Recall asks how many labeled successes are detected. Balanced accuracy accounts for unequal class sizes. AUC measures ranking rather than yes/no accuracy; Brier score and log loss measure probability error (lower is better).

The constant-training-rate baseline uses only the training success rate, so at this threshold it can predict the same outcome for everyone. Beating its ranking does not guarantee beating its accuracy or probability error. The 0.5 threshold is fixed beforehand; tuning it on this test set would invalidate the held-out comparison. Scores compare the same test rows within each scenario. Label scenarios use different definitions/populations and are not interchangeable.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display,Markdown,Image as KellyScoreImage
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,balanced_accuracy_score,roc_auc_score,brier_score_loss,log_loss

score_root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p / "data/results/kelly-version/test-predictions.csv").is_file())
score_dir=score_root / "data/results/kelly-version"
score_predictions=pd.read_csv(score_dir / "test-predictions.csv",dtype={"lease_id":str})
score_model_names=["constant_training_rate","context_only","context_plus_parking","parking_x_tenant_share"]
score_rows=[]
for scenario,g in score_predictions.groupby("scenario",sort=False):
    y=g.observed_inferred_label.to_numpy(dtype=int)
    for model in score_model_names:
        if model not in g or g[model].isna().any():continue
        probabilities=g[model].to_numpy();predicted=(probabilities>=.5).astype(int)
        tn,fp,fn,tp=confusion_matrix(y,predicted,labels=[0,1]).ravel()
        score_rows.append({"scenario":scenario,"model":model,"events":len(y),"correct":int((y==predicted).sum()),"accuracy_pct":100*(y==predicted).mean(),"balanced_accuracy_pct":100*balanced_accuracy_score(y,predicted),"precision_success_pct":100*precision_score(y,predicted,zero_division=0),"recall_success_pct":100*recall_score(y,predicted,zero_division=0),"f1_success":f1_score(y,predicted,zero_division=0),"true_negatives":int(tn),"false_positives":int(fp),"false_negatives":int(fn),"true_positives":int(tp),"auc":roc_auc_score(y,probabilities),"brier":brier_score_loss(y,probabilities),"log_loss":log_loss(y,probabilities,labels=[0,1])})
kelly_correctness=pd.DataFrame(score_rows)
display(kelly_correctness[["scenario","model","events","correct","accuracy_pct","balanced_accuracy_pct","precision_success_pct","recall_success_pct","auc","brier"]].round(3))
kelly_correctness.to_csv(score_dir / "heldout-correctness-audit.csv",index=False)
score_comparison=[]
for scenario,g in kelly_correctness.groupby("scenario",sort=False):
    baseline=g.loc[g.model.eq("context_only")].iloc[0]
    for row in g.loc[g.model.isin(["context_plus_parking","parking_x_tenant_share"])].itertuples():
        score_comparison.append({"scenario":scenario,"comparison":row.model+" versus context_only","additional_correct_events":int(row.correct-baseline.correct),"accuracy_change_percentage_points":row.accuracy_pct-baseline.accuracy_pct,"auc_change":row.auc-baseline.auc,"brier_change_lower_is_better":row.brier-baseline.brier})
kelly_correctness_comparison=pd.DataFrame(score_comparison)
display(kelly_correctness_comparison.round(4))
kelly_correctness_comparison.to_csv(score_dir / "parking-incremental-correctness.csv",index=False)

### Inspect the errors and probability calibration
Change `SCORE_SCENARIO` below to one of the three scenario names to inspect its held-out results. The confusion plots show where errors occur; the calibration plot compares average forecast probability with observed inferred success frequency. The diagonal is ideal agreement, not a fitted model. Bins with few examples are unstable.

A preview of the most confident errors is shown for diagnosis, not as an unbiased description of all predictions. No model or threshold is selected based on this preview. Repeated buildings, uncertain labels and previously inspected periods still limit conclusions.

In [ ]:
import matplotlib.pyplot as plt
SCORE_SCENARIO="app_space_retention"
# Alternatives: "tenant_retention_including_downsize", "app_excluding_open_assumptions"
score_sample=score_predictions.loc[score_predictions.scenario.eq(SCORE_SCENARIO)].copy()
if score_sample.empty:raise ValueError("No saved predictions for SCORE_SCENARIO")
score_y=score_sample.observed_inferred_label.to_numpy(dtype=int)
labels={"constant_training_rate":"Constant training rate","context_only":"Lease/property context","context_plus_parking":"Context + parking","parking_x_tenant_share":"Parking x tenant share"}
fig,axes=plt.subplots(1,4,figsize=(14,4),layout="constrained")
for ax,model in zip(axes,score_model_names):
    predicted=(score_sample[model].to_numpy()>=.5).astype(int)
    matrix=confusion_matrix(score_y,predicted,labels=[0,1])
    ax.imshow(matrix,cmap="Blues",vmin=0,vmax=len(score_y))
    for (r,c),value in np.ndenumerate(matrix):ax.text(c,r,str(value),ha="center",va="center",fontsize=15,color="black")
    ax.set(xticks=[0,1],xticklabels=["No","Yes"],yticks=[0,1],yticklabels=["No","Yes"],xlabel="Predicted success",ylabel="Inferred source success",title=labels[model])
fig.suptitle(f"Held-out predictions at threshold 0.5: {SCORE_SCENARIO} (n={len(score_y)})",fontsize=12)
confusion_file=score_dir / "kelly-confusion-matrices.png"
fig.savefig(confusion_file,dpi=170,bbox_inches="tight",facecolor="white")
display(KellyScoreImage(filename=str(confusion_file)));plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout="constrained")
calibration_rows=[]
for model in score_model_names:
    calibration=pd.DataFrame({"probability":score_sample[model].to_numpy(),"label":score_y})
    calibration["bin"]=pd.cut(calibration.probability,np.linspace(0,1,6),include_lowest=True)
    grouped=calibration.groupby("bin",observed=True).agg(events=("label","size"),average_probability=("probability","mean"),observed_frequency=("label","mean")).reset_index()
    axes[0].plot(grouped.average_probability,grouped.observed_frequency,"o-",label=labels[model])
    for row in grouped.itertuples():calibration_rows.append({"scenario":SCORE_SCENARIO,"model":model,"probability_bin":str(row.bin),"events":row.events,"average_probability":row.average_probability,"observed_frequency":row.observed_frequency})
axes[0].plot([0,1],[0,1],"--",color="gray")
axes[0].set(xlim=(0,1),ylim=(0,1),xlabel="Average predicted probability",ylabel="Observed inferred success rate",title="Are the probabilities calibrated?")
axes[0].legend(fontsize=8)
selected_metrics=kelly_correctness.loc[kelly_correctness.scenario.eq(SCORE_SCENARIO)].set_index("model").reindex(score_model_names)
axes[1].bar(range(4),selected_metrics.accuracy_pct,color=["#999999","#6A8F83","#356E99","#B56B42"])
axes[1].set(xticks=range(4),xticklabels=["Constant","Context","+ Parking","+ Interaction"],ylim=(0,100),ylabel="Correct decisions (%)",title="Does extra complexity improve yes/no decisions?")
axes[1].tick_params(axis="x",rotation=20)
for i,row in enumerate(selected_metrics.itertuples()):axes[1].text(i,row.accuracy_pct+2,f"{row.correct}/{row.events}",ha="center",fontsize=9)
calibration_file=score_dir / "kelly-calibration-and-accuracy.png"
fig.savefig(calibration_file,dpi=170,bbox_inches="tight",facecolor="white")
display(KellyScoreImage(filename=str(calibration_file)));plt.close(fig)
kelly_probability_calibration=pd.DataFrame(calibration_rows)
display(kelly_probability_calibration.round(3))
kelly_probability_calibration.to_csv(score_dir / "heldout-probability-calibration.csv",index=False)
score_errors=score_sample.copy()
score_errors["parking_prediction"]=(score_errors.context_plus_parking>=.5).astype(int)
score_errors["parking_correct"]=score_errors.parking_prediction.eq(score_errors.observed_inferred_label)
score_errors["confidence_in_decision"]=np.maximum(score_errors.context_plus_parking,1-score_errors.context_plus_parking)
display(score_errors.loc[~score_errors.parking_correct].sort_values("confidence_in_decision",ascending=False).head(10)[["lease_id","building_id","prediction_date","observed_inferred_label","context_plus_parking","parking_prediction","confidence_in_decision"]])
print("Correctness here means agreement with inferred source labels, not independently confirmed renewal contracts.")

## 8. Migration experiment: does local population outflow help?
This is a separate experiment; earlier Kelly results remain the no-migration baseline. We add the Census **net domestic migration rate per 1,000 average residents**, not gross out-migration or international emigration. Negative means domestic departures exceed arrivals. Positive means net inflow. Net values do not reveal the total number leaving.

Buildings are mapped by coordinates to official Census county polygons, not by CoStar county ID. All 91 Kelly buildings fall in Harris County or Los Angeles County. These are county-level context variables; repeating them across events does not create independent migration observations.

We use archived vintages 2016-2019 and 2021-2024, taking each vintage's latest annual rate. Availability is conservatively the day after the public Census directory file timestamp, not an independently verified first-release date. Later vintages are never joined before that proxy availability date. No 2020 vintage is used; the 2019 rate persists until a newer eligible vintage exists. A current CSV download cannot independently rule out later corrections. The 2023 generalized county boundaries are used consistently, not reconstructed historically.

Sources: [Census county vintage directories](https://www2.census.gov/programs-surveys/popest/datasets/), [official county boundary files](https://www2.census.gov/geo/tiger/GENZ2023/shp/), and [Census migration-rate definitions](https://www2.census.gov/programs-surveys/popest/technical-documentation/file-layouts/2020-2024/CO-EST2024-ALLDATA.pdf). Full URLs, hashes and availability proxies are saved locally.

In [ ]:
from pathlib import Path
import sys,subprocess,json
import numpy as np
import pandas as pd
from IPython.display import display,Markdown,Image as MigrationImage

migration_root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p / "costar-hackathon-kit/tools/kelly_migration.py").is_file())
migration_dir=migration_root / "data/results/kelly-version/migration-experiment"
# Leave False to use the completed cached preparation. True can download public files.
REFRESH_MIGRATION_INPUTS=False
if REFRESH_MIGRATION_INPUTS or not (migration_dir / "migration-matched-events.csv").is_file():
    prepared=subprocess.run([sys.executable,"-X","utf8",str(migration_root / "costar-hackathon-kit/tools/kelly_migration.py"),"--root",str(migration_root)],cwd=migration_root,capture_output=True,text=True,encoding="utf-8")
    print(prepared.stdout)
    if prepared.returncode:raise RuntimeError(prepared.stderr)
migration_data=pd.read_csv(migration_dir / "migration-matched-events.csv",dtype={"county_fips":str},parse_dates=["prediction_date","expiry_date","migration_reference_date","migration_available_date"])
migration_history=pd.read_csv(migration_dir / "county-migration-vintages.csv",dtype={"county_fips":str},parse_dates=["reference_date","available_date"])
migration_provenance=json.loads((migration_dir / "migration-provenance.json").read_text(encoding="utf-8"))
assert migration_data.migration_rate.notna().all()
assert migration_data.migration_available_date.le(migration_data.prediction_date).all()
assert migration_data.migration_reference_date.le(migration_data.prediction_date).all()
display(migration_data.groupby(["county_fips","county_name"]).agg(events=("lease_id","size"),buildings=("building_id","nunique"),earliest_prediction=("prediction_date","min"),latest_prediction=("prediction_date","max")).reset_index())
display(migration_history[["county_name","vintage","available_date","net_domestic_migration_per_1000"]])
display(migration_data[["building_id","prediction_date","county_name","migration_rate","migration_reference_date","migration_available_date","migration_vintage"]].head(12))

### Hold everything else constant
Use the same label definitions, forecasting lead, train/test gap, rows, preprocessing and fixed regularization as the Kelly baseline. Compare context only, context + migration, context + parking, and context + parking + migration. We verify that the re-created no-migration predictions match the earlier saved predictions.

This tests the incremental feature on the existing 127-event primary test, not a new untouched holdout. Migration can proxy year, geography, COVID-era changes or other economic conditions. An improvement does not prove that residents leaving caused tenant behavior, or that parking became informative. No threshold tuning or deployment is performed.

In [ ]:
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,brier_score_loss,log_loss,accuracy_score,precision_score,recall_score,balanced_accuracy_score

migration_train_base=migration_data.loc[migration_data.expiry_date.between("2019-01-01","2022-12-31")].copy()
migration_test_base=migration_data.loc[migration_data.prediction_date.ge("2023-04-01") & migration_data.expiry_date.le("2026-12-31")].copy()
migration_categories=["city","property_type","building_class","tenant_industry"]
migration_context=["building_age_years","years_since_renovation","lease_duration_years","lease_share","largest_active_tenant_share","log_building_sqft"]
migration_parking=["usage","trend","n_captures"]
migration_scenarios={"app_space_retention":("target_space_retained",False),"tenant_retention_including_downsize":("target_tenant_retained",False),"app_excluding_open_assumptions":("target_space_retained",True)}
migration_metric_rows=[];migration_predictions=[];migration_fit_rows=[]
for scenario,(target,exclude_open) in migration_scenarios.items():
    train=migration_train_base.copy();test=migration_test_base.copy()
    if exclude_open:
        train=train.loc[~train.open_occupancy_assumption.astype(bool)];test=test.loc[~test.open_occupancy_assumption.astype(bool)]
    if train[target].nunique()<2 or test[target].nunique()<2:raise ValueError("Not enough outcome classes for "+scenario)
    context=[c for c in migration_context if train[c].notna().sum()>=10]
    parking=[c for c in migration_parking if train[c].notna().sum()>=10]
    specs={"context_only":context,"context_plus_migration":context+["migration_rate"],"context_plus_parking":context+parking,"context_parking_migration":context+parking+["migration_rate"]}
    predictions=pd.DataFrame({"scenario":scenario,"lease_id":test.lease_id.astype(str).to_numpy(),"building_id":test.building_id.to_numpy(),"county_name":test.county_name.to_numpy(),"observed_inferred_label":test[target].to_numpy()})
    predictions["constant_training_rate"]=float(train[target].mean())
    for name,numeric in specs.items():
        features=ColumnTransformer([
            ("numeric",Pipeline([("impute",SimpleImputer(strategy="median",add_indicator=True)),("scale",StandardScaler())]),numeric),
            ("categorical",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=10))]),migration_categories),
        ])
        model=Pipeline([("features",features),("model",LogisticRegression(C=1.0,max_iter=3000,solver="lbfgs",random_state=42))])
        with warnings.catch_warnings(record=True) as fit_warnings:
            warnings.simplefilter("always")
            model.fit(train[numeric+migration_categories],train[target])
            predictions[name]=model.predict_proba(test[numeric+migration_categories])[:,1]
        migration_fit_rows.append({"scenario":scenario,"model":name,"train_events":len(train),"test_events":len(test),"iterations":int(model.named_steps["model"].n_iter_.max()),"converged_by_iteration_limit":bool(model.named_steps["model"].n_iter_.max()<3000),"warnings":"; ".join(dict.fromkeys(str(w.message) for w in fit_warnings))})
    for name in ["constant_training_rate",*specs]:
        y=predictions.observed_inferred_label;prob=predictions[name];pred=(prob>=.5).astype(int)
        migration_metric_rows.append({"scenario":scenario,"model":name,"test_events":len(y),"correct":int(pred.eq(y).sum()),"accuracy_pct":100*accuracy_score(y,pred),"balanced_accuracy_pct":100*balanced_accuracy_score(y,pred),"precision_success_pct":100*precision_score(y,pred,zero_division=0),"recall_success_pct":100*recall_score(y,pred,zero_division=0),"auc":roc_auc_score(y,prob),"brier":brier_score_loss(y,prob),"log_loss":log_loss(y,prob,labels=[0,1])})
    migration_predictions.append(predictions)
migration_metrics=pd.DataFrame(migration_metric_rows)
migration_test_predictions=pd.concat(migration_predictions,ignore_index=True)
migration_fit_status=pd.DataFrame(migration_fit_rows)
original_kelly=pd.read_csv(migration_root / "data/results/kelly-version/test-predictions.csv")
original_kelly["lease_id"]=original_kelly.lease_id.astype(str)
reproduction=migration_test_predictions.merge(original_kelly,on=["scenario","lease_id"],suffixes=("_rerun","_original"),validate="one_to_one")
assert len(reproduction)==len(migration_test_predictions)
for name in ["context_only","context_plus_parking"]:
    assert np.allclose(reproduction[name+"_rerun"],reproduction[name+"_original"],atol=1e-8)
print("No-migration baseline predictions match the original Kelly run on the same rows.")
display(migration_fit_status)
display(migration_metrics.round(4))
migration_deltas=[]
for scenario,g in migration_metrics.groupby("scenario",sort=False):
    indexed=g.set_index("model")
    for old,new in [("context_only","context_plus_migration"),("context_plus_parking","context_parking_migration"),("context_plus_migration","context_parking_migration")]:
        a,b=indexed.loc[old],indexed.loc[new]
        migration_deltas.append({"scenario":scenario,"comparison":new+" minus "+old,"additional_correct":int(b.correct-a.correct),"accuracy_change_pp":b.accuracy_pct-a.accuracy_pct,"auc_change":b.auc-a.auc,"brier_change_lower_is_better":b.brier-a.brier,"log_loss_change_lower_is_better":b.log_loss-a.log_loss})
migration_incremental=pd.DataFrame(migration_deltas)
display(migration_incremental.round(4))

### Visualize the result and preserve the limits
The migration time series shows county conditions, not tenant movements. The performance panels use identical test rows within each label scenario. An increase in accuracy can accompany worse ranking or probability estimates, so inspect all metrics.

Only two counties are represented, with a small number of county/vintage combinations. We do not claim reliable county-level significance from hundreds of lease rows, and a building-resampling interval would not resolve the two-county problem. Additional counties and an untouched future evaluation would be needed for broader claims.

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(15,4.5),layout="constrained")
for county,g in migration_history.groupby("county_name"):
    g=g.sort_values("reference_date")
    axes[0].plot(g.reference_date,g.net_domestic_migration_per_1000,"o-",label=county)
axes[0].axhline(0,color="gray",linestyle="--",linewidth=.8)
axes[0].set(title="County context: negative means net outflow",xlabel="Reference year",ylabel="Net domestic migration per 1,000")
axes[0].legend(fontsize=8)
primary=migration_metrics.loc[migration_metrics.scenario.eq("app_space_retention")].set_index("model")
order=["context_only","context_plus_migration","context_plus_parking","context_parking_migration"]
labels=["Context","+ Migration","+ Parking","+ Both"]
for ax,metric,title in [(axes[1],"auc","Same primary test: ranking"),(axes[2],"accuracy_pct","Same primary test: correct decisions")]:
    values=primary.reindex(order)[metric]
    ax.bar(range(4),values,color=["#6A8F83","#B39343","#356E99","#925F86"])
    ax.set(xticks=range(4),xticklabels=labels,title=title)
    ax.tick_params(axis="x",rotation=25)
    if metric=="auc":ax.set_ylim(0,1);ax.axhline(.5,color="gray",linestyle="--",linewidth=.8)
    else:ax.set_ylim(0,100);ax.axhline(primary.loc["constant_training_rate","accuracy_pct"],color="gray",linestyle="--",linewidth=.8)
    for i,value in enumerate(values):ax.text(i,value+(.015 if metric=="auc" else 1.5),f"{value:.3f}" if metric=="auc" else f"{value:.1f}%",ha="center",fontsize=9)
fig.suptitle("Kelly migration experiment: does county context add predictive value?",fontsize=13)
migration_plot=migration_dir / "migration-model-comparison.png"
fig.savefig(migration_plot,dpi=170,bbox_inches="tight",facecolor="white")
display(MigrationImage(filename=str(migration_plot)));plt.close(fig)
for name,frame in {"migration-model-metrics":migration_metrics,"migration-incremental-comparison":migration_incremental,"migration-test-predictions":migration_test_predictions,"migration-fit-status":migration_fit_status}.items():
    frame.to_csv(migration_dir / f"{name}.csv",index=False)
from datetime import datetime,timezone
migration_report={"created_at_utc":datetime.now(timezone.utc).isoformat(),"definition":migration_provenance["definition"],"source_provenance":migration_provenance,"baseline_reproduction_verified":True,"metrics":migration_metrics.to_dict("records"),"incremental_comparisons":migration_incremental.to_dict("records"),"limitations":["Two counties only","Previously inspected test period","Net migration is not gross emigration or individual tenant movement","Conservative availability proxies and possible historical-file corrections","Retrospective lease labels and existing CV measurements remain unresolved"]}
(migration_dir / "migration-model-results.json").write_text(json.dumps(migration_report,indent=2,default=str),encoding="utf-8")
print("Migration experiment saved separately; earlier Kelly results and dashboard unchanged.")